# Machine Learning Exercise

Now it's time to practice what you have seen in the previous notebooks. Your task for today is to download the data from the database and train a model in order to predict the binary label in this heart-related teaching dataset. 

![](https://www.nicepng.com/png/detail/397-3975460_disease-high-quality-png-heart-disease-cartoon-png.png)

Complete the exercise first, then compare your choices. This is not a clinically validated risk model.

## Step 1 - Define the Goal and Research Question

1. Import the data from the database. The schema is called `heart`. You can use DBeaver to get an overview over the different tables and think about a good way to join them. 
2. Conduct a brief EDA to become familiar with the data. 
3. Preprocess the data as far as you need it and...
4. ...train a logistic regression model.
Split before EDA. Use training folds for tuning, validation data for candidate selection, and the test set only once at the end.

### Success Metric

For this teaching scenario, assume missing a positive case matters more than raising a false alarm. We choose **F2 for class 1**, alongside precision, recall, accuracy, and the confusion matrix.

$$F_2 = \frac{5PR}{4P + R} = \frac{5TP}{5TP + 4FN + FP}$$

`P` is precision, `R` is recall, and TP/FN/FP are true-positive, false-negative, and false-positive counts.

The formula gives false negatives four times the coefficient of false positives; F2 is **not** a direct estimate of clinical costs. Confirm class 1's meaning with the coach before interpretation. Real screening criteria require clinical expertise and external validation.

[Reference: scikit-learn F-beta score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

### What to Use and Keep in Mind
 
* **Scale your data:** Which scaler works best in your case?
* **Tune your model:** Tune the hyperparameter of your model. You can start with a larger parameter grid and a `RandomizedSearchCV` and continue with a narrower parameter grid for your `GridSearchCV`.
* **Choose the right evaluation metric!**


### Data Overview

The database is a teaching adaptation. Confirm its label definition and category coding using the database documentation: the column name `heart_attack` alone does not establish whether it represents disease, past events, or a recoded label.

| Column | Meaning / treatment |
|---|---|
| patient_id | Join key, not a model input. Check one row per patient. |
| age | Age in years. |
| sex | Coded sex variable; check the database's coding. |
| chest_pain_type | Categorical chest-pain code; do not assume equally spaced values. |
| resting_blood_pressure | Resting blood pressure in mm Hg. |
| fasting_blood_sugar | Indicator for fasting blood sugar above 120 mg/dl; verify coding. |
| thal | Categorical test-result code; verify normal/defect/missing codes. |
| serum_cholestoral | Serum cholesterol in mg/dl; spelling follows the schema. |
| resting_electrocardiographic_results | Categorical resting ECG result. |
| maximum_heartrate_achieved | Maximum heart rate during an exercise test. |
| exercise_induced_angina | Categorical exercise-induced angina indicator. |
| oldpeak | Exercise-induced ST depression relative to rest. |
| slope_of_the_peak_exercise_st_segment | Categorical ST-segment slope code. |
| number_of_major_vessels_colored_by_flourosopy | Vessel count; verify missing/sentinel codes. |
| real_data | Provenance tag, not a predictive feature. |
| heart_attack | Binary exercise label; confirm what 0 and 1 mean before interpretation. |

[UCI Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease) describes **presence of heart disease**, not future heart-attack risk. Its original target is 0-4; public adaptations often binarize or recode it. Do not assume this database uses the original codes. This exercise is not a clinical decision tool.

The categorical descriptions are still useful: chest-pain categories describe typical/atypical angina, non-anginal pain, or no symptoms; ECG categories describe normal or abnormal resting results; ST-slope categories describe upsloping, flat, or downsloping segments. Their integer mappings must come from this database's documentation, not an assumed mapping from another dataset.

## Step 2 - Get the Data

We import the libraries we need, then pull the heart data from the database.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy.stats import loguniform, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sqlalchemy import URL, create_engine, text

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
pd.options.display.float_format = "{:,.2f}".format
RSEED = 42

### Load from the Database

The data for this exercise is stored in our postgres database in the schema `heart`. The different features are split thematically into five different tables. Your first task will be to have a look at the tables (e.g. in DBeaver) and figure out a way to join the information you need. As soon as you're happy with your query, you can use the following code cells to import the data into this notebook. 

In previous notebooks you've seen two different approaches to import data from a database into a notebook. The following code uses `sqlalchemy` in combination with pandas `pd.read_sql()`. Copy `.env.example` to `.env` in this repository root and fill in `DATABASE`, `USER_DB`, `PASSWORD`, `HOST`, and `PORT` using the provided credentials. The notebook builds the connection URL automatically; enter the password as-is, without URL encoding. Existing `.env` files with a complete `DB_STRING` are also supported. Change `query_string` to your own query. Never commit credentials.

The query preserves the original exercise's exclusion of one explicitly fabricated source tag. Other and missing tags are retained, but this does not verify their provenance.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # Build the URL from the fields in .env; URL.create handles special characters.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Join schema-qualified tables without changing the connection search path.
query_string = """
SELECT p.id AS patient_id, p.age, p.sex,
       cp.chest_pain_type,
       bm.serum_cholestoral, bm.fasting_blood_sugar, bm.thal,
       pva.resting_blood_pressure, pva.resting_electrocardiographic_results,
       pva.maximum_heartrate_achieved, pva.exercise_induced_angina, pva.oldpeak,
       pva.slope_of_the_peak_exercise_st_segment,
       pva.number_of_major_vessels_colored_by_flourosopy, pva.real_data,
       hah.heart_attack
FROM heart.patient AS p
LEFT JOIN heart.chest_pain AS cp ON p.id = cp.patient_id
LEFT JOIN heart.blood_metrics AS bm ON p.id = bm.patient_id
LEFT JOIN heart.pressure_vessels_angina AS pva ON p.id = pva.patient_id
LEFT JOIN heart.heart_attack_history AS hah ON p.id = hah.patient_id
WHERE pva.real_data <> 'Evgeny likes white wine for lunch and red wine for dinner'
   OR pva.real_data IS NULL
ORDER BY p.id
"""
# Close the connection after reading; the DataFrame remains available.
with db.connect() as connection:
    df_sqlalchemy = pd.read_sql(text(query_string), connection)
db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optional export: create the ignored data directory only when needed.
# (PROJECT_ROOT / "data").mkdir(exist_ok=True)

In [ ]:
# Optional: save the completed join locally, never to a tracked file.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

In [ ]:
# Continue directly from the query result; no CSV round trip is needed.
df = df_sqlalchemy.copy()

In [ ]:
df.head()

In [ ]:
print(f"Eligible observations: {df.shape[0]}")
print(f"Columns (including identifier, provenance and target): {df.shape[1]}")

### Structural Cleanup (before the split)

We first do structural cleaning that learns nothing from the data, so it is safe to apply before splitting: renaming the columns, and checking for and removing exact duplicate rows. Removing duplicates here (before the split) ensures the same observation cannot end up in both the train and test sets.

After removing exact duplicates, check that the join contains one row per patient. Missing target labels cannot be used for supervised training; report their count and exclude those rows. Missing input features will be handled later during preprocessing.

Do not silently deduplicate conflicting records for the same patient: fix the join or define a patient-grouped split for genuinely repeated measurements.

In [ ]:
# Renaming the column names

df.rename(
    columns={
        "fasting_blood_sugar": "fasting_bs",
        "resting_blood_pressure": "resting__bp",
        "resting_electrocardiographic_results": "electrocardiography",
        "maximum_heartrate_achieved": "max_heartrate",
        "exercise_induced_angina": "exercise_ind_angina",
        "slope_of_the_peak_exercise_st_segment": "slope_of_peak_exercise",
        "number_of_major_vessels_colored_by_flourosopy": "major_vessels_colored",
    },
    inplace=True,
)

In [ ]:
# Checking for duplicates


def check_duplicates(data):
    has_dup = data.duplicated()
    n_dup = has_dup.sum()

    if n_dup > 0:
        print("Data has", n_dup, "duplicates")
    else:
        print("Data has no duplicates")


check_duplicates(df)
print(f"Missing target labels: {int(df['heart_attack'].isna().sum())}")

In [ ]:
# Remove exact duplicates and unlabeled rows before splitting.
# Missing feature values are retained at this stage.
before = len(df)
df = df.drop_duplicates().dropna(subset=["heart_attack"]).copy()
if df["patient_id"].isna().any() or df["patient_id"].duplicated().any():
    raise ValueError(
        "The join must have one nonmissing patient_id per row. Fix the source join first."
    )
if set(df["heart_attack"].unique()) != {0, 1}:
    raise ValueError(
        "Confirm target coding: this solution expects both binary labels 0 and 1."
    )
df["heart_attack"] = df["heart_attack"].astype(int)
print(f"Dropped {before - len(df)} duplicate or unlabeled rows; {len(df)} rows remain.")

## Step 3 - Train / Validation / Test Split

Reserve 20% for the final test. Split the remainder into training and validation (60% / 20% of the full sample). Stratification preserves approximate class proportions. This assumes independent patients from the same source population.

EDA and CV searches use training data only. The validation set compares candidates; the test set remains untouched until Step 9.

The second split takes 25% of the remaining 80%, yielding approximately 60% training, 20% validation, and 20% test overall.

<img src="../assets/heart-evaluation-protocol.svg" width="650" alt="Heart exercise: training with internal cross-validation, separate validation for candidate selection, refit on development data, and one final test evaluation.">

Unlike the Titanic notebook, this solution uses a separate validation set after training-only cross-validation. Both approaches are valid; the shared rule is that test data never selects the model. Percentages below describe this exercise, not a universal requirement.

In [ ]:
development_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RSEED, stratify=df["heart_attack"]
)
train_df, val_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=RSEED,
    stratify=development_df["heart_attack"],
)
print("Training / validation / test:", len(train_df), len(val_df), len(test_df))

## Step 4 - Data Exploration

All exploration below runs on `train_df`. If you split your features and target separately instead, you can rebuild a single frame for EDA with `pd.concat([X_train, y_train], axis=1)`.

### Understand the Basics

In [ ]:
# Type and non-null counts
train_df.info()

In [ ]:
# Descriptive statistics
train_df.describe().T

In [ ]:
# Unique values per column
train_df.nunique()

### Check for Missing Values

In [ ]:
# Share of missing values per column
train_df.isnull().mean()

In [ ]:
# Class balance of the target
train_df["heart_attack"].value_counts()

### Visualize the Data

In [ ]:
# Plot all predictive features and the target, excluding identifiers and provenance.
features = [c for c in train_df.columns if c not in ("patient_id", "real_data")]
nrows = (len(features) + 2) // 3
fig, axes = plt.subplots(nrows, 3, figsize=(15, 3 * nrows), squeeze=False)
for item, ax in zip(features, axes.flat):
    # Counts remain meaningful for category codes; a density curve would imply continuity.
    sns.histplot(train_df[item], ax=ax, color="#33658A").set(title=item, xlabel="")
for ax in list(axes.flat)[len(features) :]:
    ax.set_visible(False)
fig.tight_layout()

Read the class counts above and compute the proportions. Do not assume the live database is balanced or has the same distribution as an earlier run.

### Identify Outliers

In [ ]:
# Inspect outliers in continuous measurements, not arbitrary category codes.
continuous_features = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
]
train_df[continuous_features].plot(
    kind="box", subplots=True, layout=(2, 3), figsize=(12, 7)
)
plt.tight_layout()
plt.show()

### Missing Values and Provenance

Inspect missing measurements and compare their distributions by source tag. Keep the original tags unchanged: use a separate plotting copy to display missing tags. Similar-looking distributions cannot establish that records have the same provenance.

In [ ]:
# Inspect missing features without overwriting the original provenance tags.
print(train_df[continuous_features].isna().mean().rename("missing_fraction"))
provenance_plot = train_df.copy()
provenance_plot["source_label"] = train_df["real_data"].fillna("Missing source tag")

In [ ]:
# Compare source groups using measurements, not IDs or arbitrary categorical codes.
sns.pairplot(
    provenance_plot, vars=continuous_features, hue="source_label", corner=True, height=2
);

Similar-looking distributions do not prove records have the same provenance. The original exercise's exclusion rule was applied before splitting, but remaining source tags still need confirmation. Missing features are handled later by training-fitted imputers.

Inspect how the selected measurements differ between the two label classes. These plots are exploratory, not evidence of a causal relationship.

In [ ]:
# Keep the graphical comparison by target class as well as the numeric summary.
sns.pairplot(
    train_df, vars=continuous_features, hue="heart_attack", corner=True, height=2
)
plt.show()
train_df.groupby("heart_attack")[continuous_features].median()

Describe the relationships visible in this training sample. A higher or lower measurement in one label group does not establish future infarction risk.

### Correlation Analysis

In [ ]:
# Correlation heatmap

fig, ax = plt.subplots(figsize=(12, 8))

# Select measurements and the binary target; exclude IDs and category codes
df_numeric = train_df[continuous_features + ["heart_attack"]]

# Compute correlations
correlations = df_numeric.corr()

# Generate a mask for the upper triangle
mask = np.zeros_like(correlations)
mask[np.triu_indices_from(mask)] = True

# Generate a custom diverging colormap
cmap = sns.diverging_palette(140, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio

sns.heatmap(
    correlations,
    mask=mask,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    linewidths=1,
    cbar_kws={"shrink": 0.7},
    ax=ax,
);

This heatmap shows Pearson associations in the training sample, not causal effects. The binary target can be included, but arbitrary numerical category codes are not continuous measurements. Patient identifiers are excluded and investigated separately below.

## Step 5 - Feature Engineering

Retain missing input values for imputation inside the pipeline. Exclude `patient_id` and `real_data` from predictive features. Do not remove difficult validation or test rows merely because measurements are missing.

Median imputation is used for numerical inputs; categorical inputs use their most frequent training value. Confirm any special missing-value codes first. All preprocessing is refitted within each CV training fold. `keep_empty_features=True` preserves columns that are entirely missing in a fold; investigate such columns rather than interpreting their fallback value as a measurement.

In [ ]:
# Retain missing inputs: the pipeline below will learn replacement values.
print("Training / validation / test rows:", len(train_df), len(val_df), len(test_df))

### Define Features and Target

In [ ]:
# Group columns by treatment, excluding identifiers and provenance from predictions.
num_cols = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
    "major_vessels_colored",
]
cat_cols = [
    "sex",
    "fasting_bs",
    "exercise_ind_angina",
    "chest_pain_type",
    "thal",
    "electrocardiography",
    "slope_of_peak_exercise",
]
feature_cols = num_cols + cat_cols

X_train, y_train = train_df[feature_cols], train_df["heart_attack"]
X_val, y_val = val_df[feature_cols], val_df["heart_attack"]

# Only for the deliberately flawed identifier demonstration.
X_train_id = train_df[feature_cols + ["patient_id"]]
X_val_id = val_df[feature_cols + ["patient_id"]]
print("Training / validation shapes:", X_train.shape, X_val.shape)

### Build the Feature-Engineering Pipeline

Following the first notebook, all feature engineering goes into a single `ColumnTransformer` that imputes and scales the numeric columns and imputes and one-hot encodes the categorical ones. We wrap it with the model in a `Pipeline`, so each transformer is fit on the training folds only, even inside cross-validation. The helper below builds that pipeline for a given scaler.

In [ ]:
def build_pipeline(scaler="passthrough", numeric=None):
    if numeric is None:
        numeric = num_cols
    # Fit imputation before scaling within each training fold.
    numerical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", scaler),
        ]
    )
    # Fill missing categories before encoding; learn both steps on training folds.
    categorical = Pipeline(
        [
            (
                "impute",
                SimpleImputer(strategy="most_frequent", keep_empty_features=True),
            ),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    features = ColumnTransformer(
        [
            ("num", numerical, numeric),
            ("cat", categorical, cat_cols),
        ]
    )
    return Pipeline(
        [
            ("features", features),
            ("model", LogisticRegression(max_iter=10000, random_state=RSEED)),
        ]
    )

All one-hot columns are retained for this regularized model. Unknown categories become an all-zero block with `handle_unknown="ignore"`, rather than being confused with an intentionally dropped reference category. This avoids the dropped-category warning, but unknown values still deserve investigation in a real system.

## Steps 6 to 8 - Model Training and Hyperparameter Tuning

Feature engineering, model training, hyperparameter tuning, and validation form an iterative loop. We start from a simple baseline, then improve it with scaling and tuned hyperparameters.

In [ ]:
# Keep comparable validation metrics, then inspect false positives and false negatives.
validation_results = {}


def evaluate(name, y_true, y_pred):
    metrics = {
        "F2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
    }
    f2 = metrics["F2"]
    print(f"{name}: accuracy {accuracy_score(y_true, y_pred):.2f} | F2 {f2:.2f}")
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(
        confusion_matrix(y_true, y_pred, labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="YlGnBu",
        xticklabels=[0, 1],
        yticklabels=[0, 1],
        ax=ax,
    )
    ax.set(title=name, xlabel="Predicted label", ylabel="Actual label")
    fig.tight_layout()
    plt.show()
    return metrics

### Baseline Models

Keep the original simple `max_heartrate` rule: predict the class whose training mean is nearer. The boundary is the midpoint of the two class means; its direction is learned from the training labels rather than assuming a higher heart rate means higher risk. Missing measurements use the training median. This is a teaching heuristic, not a clinical rule.

Also compare a constant-positive classifier: recall is 1 when positives are present, but every negative becomes a false positive. This extra reference shows why a high F2 alone is not enough.

In [ ]:
def predict_heartrate_rule(training, rows):
    """Learn a midpoint rule from training data and apply it to new rows."""
    observed = training["max_heartrate"].dropna()
    if observed.empty:
        raise ValueError("The heart-rate rule needs observed training measurements.")
    # Learn replacements and the class boundary from training observations only.
    fill_value = observed.median()
    class_means = (
        training["max_heartrate"]
        .fillna(fill_value)
        .groupby(training["heart_attack"])
        .mean()
    )
    print("Training class mean heart rates:", class_means.round(1).to_dict())
    print("Midpoint threshold:", round(class_means.mean(), 1))
    values = rows["max_heartrate"].fillna(fill_value)
    # Nearest class mean; exact ties predict class 0.
    return ((values - class_means[1]).abs() < (values - class_means[0]).abs()).astype(
        int
    )


validation_results["Heart-rate rule"] = evaluate(
    "Heart-rate rule (validation)", y_val, predict_heartrate_rule(train_df, X_val)
)

baseline = DummyClassifier(strategy="constant", constant=1)
baseline.fit(X_train, y_train)
validation_results["Constant positive baseline"] = evaluate(
    "Baseline (validation)", y_val, baseline.predict(X_val)
)

### First Logistic Regression (and a Leakage Trap)

Now a real model. We fit a logistic regression through the pipeline, first without `patient_id` and then with it, to show how an identifier can leak the target.

In [ ]:
# Leave numeric features unscaled; impute and encode within the pipeline.
log_reg = build_pipeline()
log_reg.fit(X_train, y_train)
validation_results["Unscaled logistic regression"] = evaluate(
    "Unscaled logistic regression (validation)", y_val, log_reg.predict(X_val)
)

In [ ]:
# Deliberately include the identifier to illustrate an invalid candidate.
log_reg_id = build_pipeline(numeric=num_cols + ["patient_id"])
log_reg_id.fit(X_train_id, y_train)
evaluate(
    "Identifier included: invalid candidate (validation)",
    y_val,
    log_reg_id.predict(X_val_id),
)

If adding `patient_id` improves performance sharply, suspect a collection or ordering artifact. An ID is not automatically leakage, but a label-correlated identifier without predictive meaning is not a defensible feature. This deliberately flawed model is excluded from selection.

In [ ]:
sns.kdeplot(data=train_df, x="patient_id", hue="heart_attack", fill=True);

Inspect whether labels occupy different ID ranges. The pattern can change when the teaching database changes, so do not assume a fixed cutoff.

In [ ]:
_ = sns.scatterplot(data=train_df, x="patient_id", y="heart_attack")

In [ ]:
# Check how many training observations belong to each class.
train_df.groupby("heart_attack")["patient_id"].count().rename("count")

In [ ]:
# Compare class-specific ID ranges without assuming a fixed database cutoff.
id_ranges = train_df.groupby("heart_attack")["patient_id"].agg(["min", "max"])
id_ranges

In [ ]:
# Overlapping ranges do not rule out ordering artifacts; use the plots too.
overlap_start = id_ranges["min"].max()
overlap_end = id_ranges["max"].min()
print("Class ID ranges overlap:", overlap_start <= overlap_end)

Database ordering can encode how records were imported or sorted. A random row split may preserve such artifacts in every subset and yield misleadingly high scores. No fixed patient-ID threshold is part of this solution.

### Training with Feature Scaling

Compare standardization with min-max scaling using the same validation set. Only the scaler changes; preprocessing and model choices otherwise remain fixed.

#### Standardization

In [ ]:
std_pipe = build_pipeline(StandardScaler())
std_pipe.fit(X_train, y_train)
validation_results["Standardized logistic regression"] = evaluate(
    "Standardized (validation)", y_val, std_pipe.predict(X_val)
)

#### Min-Max Scaling

In [ ]:
minmax_pipe = build_pipeline(MinMaxScaler())
minmax_pipe.fit(X_train, y_train)
validation_results["Min-max logistic regression"] = evaluate(
    "Min-max (validation)", y_val, minmax_pipe.predict(X_val)
)

### Hyperparameter Tuning with RandomizedSearchCV

Search the full standardized pipeline using stratified training folds. Keep the budget modest: ten sampled combinations and five folds. Vary `C` (smaller means stronger regularization) and `l1_ratio` (0 = L2, 1 = L1, intermediate values = elastic net). Use `saga`, which supports all three.

L1 can set coefficients to zero; L2 shrinks coefficients without generally setting them exactly to zero. The random search samples mixtures; the grid below explicitly compares the two endpoints.

[Reference: scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

The `model__` prefix addresses parameters of the pipeline's `model` step; preprocessing remains inside each fitted fold.

In [ ]:
# F2 gives recall more weight; inspect precision and error counts as well.
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
if y_train.value_counts().min() < cv.n_splits:
    raise ValueError("Need at least five training examples per class for five-fold CV.")

random_search = RandomizedSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": loguniform(0.01, 10),
        "model__l1_ratio": uniform(0, 1),
    },
    n_iter=10,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RSEED,
    error_score="raise",
)
random_search.fit(X_train, y_train)
print("Best parameters:", random_search.best_params_)
print(f"Best training CV F2: {random_search.best_score_:.3f}")

### Hyperparameter Tuning with GridSearchCV

Evaluate four C values for each of L2 (`l1_ratio=0`) and L1 (`l1_ratio=1`): eight combinations, using the same folds and metric as the random search. Compare **CV** results to select the tuning method before using the validation set for the final candidate comparison. Neither search guarantees the best possible model outside its search space.

These C values are evenly spaced on a logarithmic scale, equivalent to `np.logspace(-2, 1, num=4)`.

In [ ]:
# Compare L2 and L1 with the same folds and metric as randomized search.
grid_search = GridSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__l1_ratio": [0, 1],
    },
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print(f"Best training CV F2: {grid_search.best_score_:.3f}")

selected_search = max(
    [grid_search, random_search], key=lambda search: search.best_score_
)
tuned_model = selected_search.best_estimator_
validation_results["Tuned logistic regression"] = evaluate(
    "Tuned model (validation)", y_val, tuned_model.predict(X_val)
)
# Compare eligible candidates on the same validation rows; keep test results separate.
validation_comparison = pd.DataFrame.from_dict(validation_results, orient="index")
validation_comparison.sort_values("F2", ascending=False, kind="stable")

## Step 9 - Calculate Test Score

Choose the eligible candidate with the highest validation F2, including both baselines. Ties follow their insertion order in `validation_results`. Refit the selected model (or relearn the heart-rate rule) on training plus validation data, then evaluate **once** on the test set.

Search objects already refit on their training input, but they have not seen the separate validation subset. Report the final confusion matrix, precision, recall, and F2. If the baseline wins, report that outcome rather than claiming the more complex model is better.

Read precision and recall alongside F2: the constant-positive baseline has recall 1 but its precision equals the positive-class proportion. F2 favors recall, so a high F2 alone does not imply a useful model. Metrics refer to class 1; undefined ratios are reported as 0 by convention. The table is rounded for display only; selection uses full-precision F2.

In [ ]:
candidates = {
    "Constant positive baseline": baseline,
    "Unscaled logistic regression": log_reg,
    "Standardized logistic regression": std_pipe,
    "Min-max logistic regression": minmax_pipe,
    "Tuned logistic regression": tuned_model,
}
selected_name = max(validation_results, key=lambda name: validation_results[name]["F2"])
X_test, y_test = test_df[feature_cols], test_df["heart_attack"]
if selected_name == "Heart-rate rule":
    y_pred = predict_heartrate_rule(development_df, X_test)
else:
    # Refit the selected configuration on development data before the final test.
    best_model = clone(candidates[selected_name])
    best_model.fit(development_df[feature_cols], development_df["heart_attack"])
    y_pred = best_model.predict(X_test)
print(f"Selected using validation data: {selected_name}")
evaluate("Final test", y_test, y_pred)
print(classification_report(y_test, y_pred, labels=[0, 1], zero_division=0))

## Step 10 - Deployment and Limitations

This teaching model must **not** be deployed as a medical decision tool. The database's provenance and label coding need confirmation, and one small random split cannot establish clinical validity or fairness.

For a real project, define the prediction moment, confirm which measurements are available then, obtain appropriate governance review, and validate on independent representative data. Monitoring would include data quality, distribution shift, subgroup performance, and error costs, not just overall accuracy.

## Check Your Understanding - Answers

1. **Why exclude `patient_id`?** It identifies a patient rather than a generalizable measurement. It may encode collection order or source and produce a misleading shortcut. Keep it for joins and duplicate checks, not as a predictor.
2. **Why fit imputation inside each fold?** Medians or other replacement values must come only from that fold's training rows. Computing them before cross-validation lets held-out rows influence preprocessing and can bias evaluation. The pipeline enforces the boundary.
3. **Why not select using test scores?** Choosing the best test score uses the test set for model selection, making it an optimistic estimate of the selected model's performance. Freeze choices using training CV and validation, refit on development data, then evaluate once on test.
